# 1. Sample construction

This notebook builds the weekly product-level dataset used in the analysis. It combines sales, product, and store information, reconstructs missing prices where possible, creates basic calendar and promotion variables, and applies the planned data-quality checks.

The notebook creates three analytical datasets:

- `main_data`: the cleaned weekly panel;
- `demand_data`: observations suitable for demand analysis;
- `demand_model_data`: the final sample used to estimate demand.


The price reconstruction is checked against observed prices. Promotion-history variables are created using current and past information. Unusual sales observations are identified using a pre-specified rule and are reported separately.

**Inputs:** Dominick's source files. **Outputs:** processed weekly panels and sample-quality audits consumed by Notebooks 02–05.

## 1. Setup


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from dateutil.easter import easter
from IPython.display import display

from dynamic_promotion_planning.config import load_analysis_config
from dynamic_promotion_planning.paths import project_paths
from dynamic_promotion_planning.sample import (
    detect_cross_store_sales_outliers,
    fourth_thursday_of_november,
    write_parquet_outputs,
)

In [2]:
paths = project_paths()
analysis_config = load_analysis_config()
sample_config = analysis_config.sample

PROJECT_ROOT = paths.root
RAW_DIR = paths.data_raw
PROCESSED_DIR = paths.data_processed
TABLE_DIR = paths.result_tables

MOVEMENT_PATH = RAW_DIR / "wcer.csv"
PRODUCT_PATH = RAW_DIR / "upccer.csv"
STORE_PATH = RAW_DIR / "demo.dta"

for path in [MOVEMENT_PATH, PRODUCT_PATH, STORE_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Required source file not found: {path}")

In [3]:
EXCLUDED_WEEKS = set(sample_config.excluded_weeks)
NON_CEREAL_UPCS = set(sample_config.non_cereal_upcs)

DISCOUNT_THRESHOLD = sample_config.discount_threshold
REGULAR_PRICE_WINDOW = sample_config.regular_price_window
REGULAR_PRICE_MIN_PERIODS = sample_config.regular_price_min_periods
REGULAR_PRICE_QUANTILE = sample_config.regular_price_quantile

SALES_OUTLIER_MIN_MOVE = sample_config.sales_outlier_min_move
SALES_OUTLIER_MIN_STORES = sample_config.sales_outlier_min_stores
SALES_OUTLIER_MIN_P95_RATIO = sample_config.sales_outlier_min_p95_ratio
PARQUET_COMPRESSION = sample_config.parquet_compression
RANDOM_SEED = sample_config.random_seed

## 2. Load and merge source data

The movement data are merged with the UPC lookup using a many-to-one restriction. The merge audit verifies product coverage and the uniqueness of store–UPC–week observations.


In [4]:
movement = pd.read_csv(MOVEMENT_PATH)
products = pd.read_csv(PRODUCT_PATH, encoding="cp1252")
stores = pd.read_stata(STORE_PATH)

# Standardize names before downstream processing.
for frame in [movement, products, stores]:
    frame.columns = frame.columns.str.strip().str.lower()

required_movement = {
    "store", "upc", "week", "move", "qty", "price",
    "sale", "profit", "ok",
}
required_products = {"upc", "descrip", "size"}
required_stores = {"store", "priclow", "pricmed", "prichigh"}

assert required_movement.issubset(movement.columns)
assert required_products.issubset(products.columns)
assert required_stores.issubset(stores.columns)
assert not products["upc"].duplicated().any()

In [5]:
data = movement.merge(
    products,
    on="upc",
    how="left",
    validate="many_to_one",
    indicator="product_merge",
)

merge_audit = pd.Series(
    {
        "rows": len(data),
        "stores": data["store"].nunique(dropna=True),
        "products": data["upc"].nunique(dropna=True),
        "weeks": data["week"].nunique(dropna=True),
        "store_product_pairs": data[["store", "upc"]].drop_duplicates().shape[0],
        "matched_product_share": data["descrip"].notna().mean(),
        "duplicate_store_upc_week_rows": data.duplicated(
            ["store", "upc", "week"]
        ).sum(),
    },
    name="value",
)

display(merge_audit)

assert data["product_merge"].eq("both").all()
assert merge_audit["duplicate_store_upc_week_rows"] == 0

data = data.drop(columns="product_merge")

rows                             6602582.0
stores                                93.0
products                             490.0
weeks                                367.0
store_product_pairs                36620.0
matched_product_share                  1.0
duplicate_store_upc_week_rows          0.0
Name: value, dtype: float64

## 3. Standardize the data and classify stores by price level

`SALE` is treated as a recorded-promotion flag whenever it is nonmissing. Store price tiers are reconstructed from the mutually exclusive `priclow`, `pricmed`, and `prichigh` indicators. Stores without tier metadata are excluded because tier membership is required for the main price-imputation procedure.


In [6]:
# Normalize promotion codes and standardize key identifiers.
data["sale"] = (
    data["sale"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)
data["promo_recorded"] = data["sale"].notna()

data["store"] = pd.to_numeric(
    data["store"], errors="raise"
).astype("Int64")
data["week"] = pd.to_numeric(
    data["week"], errors="raise"
).astype("int16")
stores["store"] = pd.to_numeric(
    stores["store"], errors="coerce"
).astype("Int64")

assert data["week"].between(1, 399).all()

In [7]:
tier_columns = ["priclow", "pricmed", "prichigh"]
stores[tier_columns] = stores[tier_columns].apply(
    pd.to_numeric,
    errors="coerce",
)

stores["tier_indicator_sum"] = stores[tier_columns].fillna(0).sum(axis=1)
if not stores["tier_indicator_sum"].eq(1).all():
    bad_rows = stores.loc[
        ~stores["tier_indicator_sum"].eq(1),
        ["store"] + tier_columns,
    ]
    raise ValueError(f"Invalid price-tier indicators:\n{bad_rows}")

stores["price_tier"] = np.select(
    [
        stores["priclow"].eq(1),
        stores["pricmed"].eq(1),
        stores["prichigh"].eq(1),
    ],
    ["low", "medium", "high"],
    default=pd.NA,
)
stores["price_tier"] = pd.Categorical(
    stores["price_tier"],
    categories=["low", "medium", "high"],
)

C:\Users\janza\AppData\Local\Temp\ipykernel_24332\3634275374.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  stores["tier_indicator_sum"] = stores[tier_columns].fillna(0).sum(axis=1)
C:\Users\janza\AppData\Local\Temp\ipykernel_24332\3634275374.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  stores["price_tier"] = np.select(


In [8]:
# Duplicate metadata rows are allowed only if they assign the same tier.
tier_conflicts = stores.groupby(
    "store", observed=True
)["price_tier"].nunique()
assert tier_conflicts.le(1).all()

store_info = (
    stores[["store", "price_tier"]]
    .dropna(subset=["store"])
    .drop_duplicates(subset="store")
)
assert not store_info["store"].duplicated().any()

data = data.merge(
    store_info,
    on="store",
    how="left",
    validate="many_to_one",
    indicator="store_merge",
)

unmatched_stores = sorted(
    data.loc[
        data["price_tier"].isna(), "store"
    ].dropna().unique().tolist()
)

In [9]:
# Exclude invalid observations before price imputation and price-history construction.
valid_source_row = (
    data["price_tier"].notna()
    & data["ok"].eq(1)
    & ~data["week"].isin(EXCLUDED_WEEKS)
    & data["qty"].gt(0)
    & data["move"].ge(0)
    & data["store"].notna()
    & data["upc"].notna()
)

source_quality_audit = pd.Series(
    {
        "raw_merged_rows": len(data),
        "movement_stores": data["store"].nunique(),
        "stores_with_price_tier": data.loc[
            data["price_tier"].notna(), "store"
        ].nunique(),
        "rows_with_price_tier": data["price_tier"].notna().sum(),
        "rows_with_ok_1": data["ok"].eq(1).sum(),
        "valid_tier_matched_rows": valid_source_row.sum(),
    },
    name="value",
)

display(source_quality_audit)
print("Stores excluded because price-tier metadata are unavailable:", unmatched_stores)

raw_merged_rows            6602582
movement_stores                 93
stores_with_price_tier          86
rows_with_price_tier       6553278
rows_with_ok_1             6461297
valid_tier_matched_rows    6411993
Name: value, dtype: int64

Stores excluded because price-tier metadata are unavailable: [135, 140, 141, 142, 143, 144, 146]


In [10]:
# Diagnose impossible weekly input patterns before applying any exclusions.
diagnostic_data = data.assign(
    _price=pd.to_numeric(data["price"], errors="coerce"),
    _move=pd.to_numeric(data["move"], errors="coerce"),
)
weekly_input_diagnostics = (
    diagnostic_data.groupby("week", observed=True)
    .agg(
        rows=("week", "size"),
        stores=("store", "nunique"),
        upcs=("upc", "nunique"),
        total_move=("_move", "sum"),
        zero_move_share=("_move", lambda value: value.eq(0).mean()),
        zero_price_share=("_price", lambda value: value.le(0).mean()),
        positive_price_share=("_price", lambda value: value.gt(0).mean()),
        mean_positive_price=("_price", lambda value: value[value.gt(0)].mean()),
    )
    .reset_index()
)

weekly_input_diagnostics["log_total_move"] = np.log1p(
    weekly_input_diagnostics["total_move"]
)
for column in ["log_total_move", "zero_move_share", "zero_price_share"]:
    median = weekly_input_diagnostics[column].median()
    mad = (weekly_input_diagnostics[column] - median).abs().median()
    weekly_input_diagnostics[f"{column}_robust_deviation"] = (
        (weekly_input_diagnostics[column] - median).abs() / max(mad, 1e-12)
    )

weekly_input_diagnostics["structural_zero_anomaly"] = (
    weekly_input_diagnostics["zero_move_share"].eq(1)
    & weekly_input_diagnostics["zero_price_share"].eq(1)
)
weekly_input_diagnostics["anomaly_score"] = weekly_input_diagnostics[
    [
        "log_total_move_robust_deviation",
        "zero_move_share_robust_deviation",
        "zero_price_share_robust_deviation",
    ]
].max(axis=1)

weekly_input_diagnostics.to_csv(
    TABLE_DIR / "weekly_input_diagnostics.csv",
    index=False,
)
display(
    weekly_input_diagnostics.sort_values(
        ["structural_zero_anomaly", "anomaly_score"],
        ascending=[False, False],
    ).head(10)
)
display(weekly_input_diagnostics.loc[
    weekly_input_diagnostics["week"].between(217, 221)
])
if not weekly_input_diagnostics.loc[
    weekly_input_diagnostics["structural_zero_anomaly"], "week"
].isin(EXCLUDED_WEEKS).all():
    raise AssertionError("A structural-zero week is not listed for exclusion.")

,week,rows,stores,upcs,total_move,zero_move_share,zero_price_share,positive_price_share,mean_positive_price,log_total_move,log_total_move_robust_deviation,zero_move_share_robust_deviation,zero_price_share_robust_deviation,structural_zero_anomaly,anomaly_score
218,219,16971,86,250,0,1.000000,1.000000,0.000000,NaN,0.000000,130.753364,24.421278,24.330962,True,130.753364
12,13,17317,82,249,876554,0.420396,0.420396,0.579604,2.663553,13.683755,13.317320,5.249819,5.230404,False,13.317320
29,30,17317,82,249,748569,0.372640,0.372640,0.627360,2.760746,13.525920,11.655542,3.670184,3.656611,False,11.655542
34,35,17317,82,249,223560,0.454409,0.454409,0.545591,2.764877,12.317440,1.068056,6.374855,6.351279,False,6.374855
33,34,17317,82,249,203388,0.447884,0.447884,0.552116,2.771104,12.222876,2.063681,6.159017,6.136239,False,6.159017
1,2,17317,82,249,212892,0.433447,0.433447,0.566553,2.605345,12.268545,1.582848,5.681497,5.660485,False,5.681497
5,6,17317,82,249,198641,0.431657,0.431657,0.568343,2.621803,12.199259,2.312326,5.622284,5.601492,False,5.622284
9,10,17317,82,249,227040,0.430155,0.430155,0.569845,2.648781,12.332886,0.905428,5.572622,5.552013,False,5.572622
15,16,17317,82,249,181898,0.426922,0.426922,0.573078,2.706672,12.111207,3.239397,5.465658,5.445445,False,5.465658
11,12,17317,82,249,247788,0.424900,0.424900,0.575100,2.656782,12.420333,0.015265,5.398805,5.378839,False,5.398805


,week,rows,stores,upcs,total_move,zero_move_share,zero_price_share,positive_price_share,mean_positive_price,log_total_move,log_total_move_robust_deviation,zero_move_share_robust_deviation,zero_price_share_robust_deviation,structural_zero_anomaly,anomaly_score
216,217,16967,86,250,338867,0.259681,0.259681,0.740319,3.293502,12.733366,3.311063,0.066143,0.065899,False,3.311063
217,218,16971,86,250,227052,0.249072,0.249072,0.750928,3.050064,12.332939,0.904872,0.417042,0.415500,False,0.904872
218,219,16971,86,250,0,1.000000,1.000000,0.000000,NaN,0.000000,130.753364,24.421278,24.330962,True,130.753364
219,220,17019,86,248,161307,0.258887,0.258887,0.741113,3.320572,11.991071,4.504260,0.092387,0.092046,False,4.504260
220,221,17019,86,248,327229,0.255597,0.255597,0.744403,3.303135,12.698419,2.943116,0.201225,0.200481,False,2.943116


In [11]:
main_data = data.loc[valid_source_row].copy()
main_data = main_data.drop(columns="store_merge")

assert main_data["ok"].eq(1).all()
assert not main_data["week"].isin(EXCLUDED_WEEKS).any()
assert main_data["qty"].gt(0).all()
assert main_data["move"].ge(0).all()

# Compact dtypes to reduce memory requirements.
main_data["store"] = main_data["store"].astype("int16")
main_data["move"] = main_data["move"].astype("int32")
main_data["sale"] = main_data["sale"].astype("category")

unused_source_columns = [
    "price_hex", "profit_hex", "com_code", "case", "nitem", "ok",
]
main_data = main_data.drop(
    columns=[c for c in unused_source_columns if c in main_data.columns]
)

del data, movement, products, stores

## 4. Price reconstruction

The observed unit price is calculated as `PRICE / QTY` when both variables are positive. For zero-sales observations without a recorded price, the price is imputed using the median price of the same UPC, week, and store price tier. Positive-sales observations with missing prices are not imputed.

The raw `PROFIT` field is retained only when the transaction price is observed. This avoids treating values attached to imputed zero-sales observations as observed profit margins.


In [12]:
# Compute unit prices only from observations with positive price and quantity.
main_data["unit_price_observed"] = (
    main_data["price"] / main_data["qty"]
).where(
    main_data["price"].gt(0)
    & main_data["qty"].gt(0)
)

price_group = main_data.groupby(
    ["upc", "week", "price_tier"],
    observed=True,
)["unit_price_observed"]

# Record the median price and number of valid donors in each group.
main_data["tier_week_median_price"] = price_group.transform("median")
main_data["tier_week_price_donors"] = (
    price_group.transform("count").astype("int16")
)

In [13]:
# Impute missing prices only for zero-sales rows with an available group median.
can_impute = (
    main_data["move"].eq(0)
    & main_data["unit_price_observed"].isna()
    & main_data["tier_week_median_price"].notna()
)

main_data["model_unit_price"] = main_data["unit_price_observed"]
main_data.loc[can_impute, "model_unit_price"] = main_data.loc[
    can_impute, "tier_week_median_price"
]

# Record whether each model price is observed, imputed, or unresolved.
main_data["price_source"] = np.select(
    [main_data["unit_price_observed"].notna(), can_impute],
    ["observed", "tier_median_imputed"],
    default="unresolved",
)
main_data["price_source"] = pd.Categorical(
    main_data["price_source"],
    categories=["observed", "tier_median_imputed", "unresolved"],
)

main_data["price_imputed"] = main_data["price_source"].eq(
    "tier_median_imputed"
)

# Retain observed margins only when the transaction price is observed.
main_data["gross_margin_pct_observed"] = main_data["profit"].where(
    main_data["unit_price_observed"].notna()
)

In [14]:
imputed = main_data["price_source"].eq("tier_median_imputed")
observed = main_data["price_source"].eq("observed")
unresolved = main_data["price_source"].eq("unresolved")

# Summarize the coverage and quality of the reconstructed prices.
price_reconstruction_audit = pd.Series(
    {
        "observed_prices": observed.sum(),
        "imputed_prices": imputed.sum(),
        "unresolved_prices": unresolved.sum(),
        "usable_price_share": main_data["model_unit_price"].notna().mean(),
        "imputed_share_with_at_least_3_donors": main_data.loc[
            imputed, "tier_week_price_donors"
        ].ge(3).mean(),
    },
    name="value",
)

display(price_reconstruction_audit)

# Verify that the reconstruction follows the intended rules.
assert main_data.loc[imputed, "move"].eq(0).all()
assert main_data.loc[imputed, "model_unit_price"].gt(0).all()
assert np.allclose(
    main_data.loc[observed, "model_unit_price"],
    main_data.loc[observed, "unit_price_observed"],
)
assert main_data.loc[unresolved, "model_unit_price"].isna().all()
assert not (imputed & main_data["move"].gt(0)).any()

# Drop the raw price fields after unit-price construction.
main_data = main_data.drop(columns=["price", "qty"])

observed_prices                         4.664930e+06
imputed_prices                          6.873050e+05
unresolved_prices                       1.059758e+06
usable_price_share                      8.347225e-01
imputed_share_with_at_least_3_donors    7.640553e-01
Name: value, dtype: float64

### 4.1 Held-out validation of tier-median imputation

One observed store price is held out from each UPC–week–tier group with at least two observed prices. The held-out value is then predicted using the median price among the remaining stores in the same group.


In [15]:
group_columns = ["upc", "week", "price_tier"]

# Keep observed prices from groups with at least two stores.
observed_prices = main_data.loc[
    main_data["unit_price_observed"].notna(),
    group_columns + ["store", "unit_price_observed"],
].copy()

observed_prices["group_size"] = observed_prices.groupby(
    group_columns,
    observed=True,
)["unit_price_observed"].transform("size")

eligible = observed_prices.loc[
    observed_prices["group_size"].ge(2)
].copy()

# Select one reproducible holdout from each eligible group.
rng = np.random.default_rng(RANDOM_SEED)
eligible["_random"] = rng.random(len(eligible))

holdout_indices = (
    eligible.sort_values("_random")
    .drop_duplicates(group_columns)
    .index
)

In [16]:
# Predict each held-out price from the remaining stores in its group.
holdout = observed_prices.loc[holdout_indices].copy()
donor_data = observed_prices.drop(index=holdout_indices)

donor_medians = (
    donor_data.groupby(group_columns, observed=True)
    .agg(
        predicted_price=("unit_price_observed", "median"),
        validation_donors=("unit_price_observed", "size"),
    )
    .reset_index()
)

validation = holdout.merge(
    donor_medians,
    on=group_columns,
    how="left",
    validate="one_to_one",
)

assert validation["predicted_price"].notna().all()
assert validation["validation_donors"].ge(1).all()

In [17]:
# Calculate prediction errors and group observations by donor count.
validation["absolute_error"] = (
    validation["predicted_price"]
    - validation["unit_price_observed"]
).abs()
validation["percentage_error"] = (
    validation["predicted_price"]
    - validation["unit_price_observed"]
) / validation["unit_price_observed"]
validation["absolute_percentage_error"] = (
    validation["percentage_error"].abs()
)

validation["donor_group"] = pd.cut(
    validation["validation_donors"],
    bins=[0, 1, 2, float("inf")],
    labels=["1 donor", "2 donors", "3+ donors"],
    include_lowest=True,
)

In [18]:
# Summarize overall validation accuracy and bias.
validation_summary = pd.Series(
    {
        "validated_prices": len(validation),
        "median_absolute_error": validation["absolute_error"].median(),
        "median_absolute_percentage_error": validation[
            "absolute_percentage_error"
        ].median(),
        "mean_absolute_percentage_error": validation[
            "absolute_percentage_error"
        ].mean(),
        "mean_percentage_error": validation["percentage_error"].mean(),
        "rmse": np.sqrt(np.mean(validation["absolute_error"] ** 2)),
        "within_5_percent": validation[
            "absolute_percentage_error"
        ].le(0.05).mean(),
        "within_10_percent": validation[
            "absolute_percentage_error"
        ].le(0.10).mean(),
        "p95_absolute_percentage_error": validation[
            "absolute_percentage_error"
        ].quantile(0.95),
        "p99_absolute_percentage_error": validation[
            "absolute_percentage_error"
        ].quantile(0.99),
    },
    name="value",
)

In [19]:
# Compare validation accuracy across donor counts and price tiers.
validation_by_donors = (
    validation.groupby("donor_group", observed=True)
    .agg(
        observations=("absolute_percentage_error", "size"),
        median_ape=("absolute_percentage_error", "median"),
        mean_ape=("absolute_percentage_error", "mean"),
        p95_ape=("absolute_percentage_error", lambda x: x.quantile(0.95)),
        within_5_percent=(
            "absolute_percentage_error",
            lambda x: x.le(0.05).mean(),
        ),
        mean_signed_error=("percentage_error", "mean"),
    )
)

validation_by_tier = (
    validation.groupby("price_tier", observed=True)
    .agg(
        observations=("absolute_percentage_error", "size"),
        median_ape=("absolute_percentage_error", "median"),
        mean_ape=("absolute_percentage_error", "mean"),
        p95_ape=("absolute_percentage_error", lambda x: x.quantile(0.95)),
        within_5_percent=(
            "absolute_percentage_error",
            lambda x: x.le(0.05).mean(),
        ),
    )
)

display(validation_summary)
display(validation_by_donors)
display(validation_by_tier)

# Remove temporary validation objects before the next processing step.
del observed_prices, eligible, holdout, donor_data, donor_medians

validated_prices                    196427.000000
median_absolute_error                    0.000000
median_absolute_percentage_error         0.000000
mean_absolute_percentage_error           0.016744
mean_percentage_error                    0.004560
rmse                                     0.123918
within_5_percent                         0.856364
within_10_percent                        0.982462
p95_absolute_percentage_error            0.079545
p99_absolute_percentage_error            0.119332
Name: value, dtype: float64

,observations,median_ape,mean_ape,p95_ape,within_5_percent,mean_signed_error
donor_group,,,,,,
1 donor,4780,0.0,0.020836,0.082609,0.875314,0.005631
2 donors,3510,0.0,0.021951,0.071711,0.905698,0.005341
3+ donors,188137,0.0,0.016543,0.079545,0.854962,0.004518


,observations,median_ape,mean_ape,p95_ape,within_5_percent
price_tier,,,,,
low,62734,0.0,0.024699,0.082508,0.755173
medium,69324,0.0,0.012258,0.074349,0.912195
high,64369,0.0,0.013822,0.079295,0.894856


## 5. Calendar controls

Dominick's weeks run from Thursday through Wednesday. Calendar month is assigned from the week-ending date. Holiday indicators equal one when the holiday date falls within the corresponding Thursday-Wednesday interval.


In [20]:
week_lookup = pd.DataFrame(
    {"week": np.arange(1, 400, dtype=np.int16)}
)
week_lookup["week_start"] = (
    pd.Timestamp("1989-09-14")
    + pd.to_timedelta((week_lookup["week"] - 1) * 7, unit="D")
)
week_lookup["week_end"] = (
    week_lookup["week_start"] + pd.Timedelta(days=6)
)
week_lookup["calendar_month"] = (
    week_lookup["week_end"].dt.month.astype("int8")
)
week_lookup["time_trend"] = (week_lookup["week"] - 1).astype("int16")

In [21]:
years = range(
    week_lookup["week_start"].dt.year.min(),
    week_lookup["week_end"].dt.year.max() + 1,
)

holiday_dates = {
    "thanksgiving_week": [
        fourth_thursday_of_november(year) for year in years
    ],
    "christmas_week": [
        pd.Timestamp(year=year, month=12, day=25) for year in years
    ],
    "new_year_week": [
        pd.Timestamp(year=year, month=1, day=1) for year in years
    ],
    "easter_week": [
        pd.Timestamp(easter(year)) for year in years
    ],
}

for variable, dates in holiday_dates.items():
    week_lookup[variable] = False
    for holiday_date in dates:
        contains_holiday = (
            week_lookup["week_start"].le(holiday_date)
            & week_lookup["week_end"].ge(holiday_date)
        )
        week_lookup.loc[contains_holiday, variable] = True

In [22]:
assert len(week_lookup) == 399
assert week_lookup["week"].is_unique
assert week_lookup.loc[
    week_lookup["week"].eq(1), "week_start"
].iloc[0] == pd.Timestamp("1989-09-14")
assert week_lookup.loc[
    week_lookup["week"].eq(399), "week_end"
].iloc[0] == pd.Timestamp("1997-05-07")
assert (
    week_lookup["week_end"] - week_lookup["week_start"]
).eq(pd.Timedelta(days=6)).all()

main_data = main_data.merge(
    week_lookup,
    on="week",
    how="left",
    validate="many_to_one",
    indicator="calendar_merge",
)
assert main_data["calendar_merge"].eq("both").all()
main_data = main_data.drop(columns="calendar_merge")

## 6. Historical prices and promotion variables

The regular-price benchmark is the 90th percentile of available model prices over the preceding 13 calendar weeks, excluding the current week. A week is classified as promotional when either the recorded `SALE` indicator is active or the observed price is at least 5% below this benchmark.

In [23]:
key_columns = ["store", "upc", "week_end"]

main_data = (
    main_data
    .drop(columns=["regular_price"], errors="ignore")
    .sort_values(key_columns)
    .reset_index(drop=True)
)

# Each store–UPC–week should identify one observation.
duplicates = main_data.duplicated(key_columns, keep=False)
assert not duplicates.any(), (
    "Duplicate store–UPC–week observations detected:\n"
    f"{main_data.loc[duplicates, key_columns].head(20)}"
)

rolling_regular_price = (
    main_data[
        key_columns + ["model_unit_price"]
    ]
    .groupby(
        ["store", "upc"],
        observed=True,
        sort=False,
    )
    .rolling(
        window=REGULAR_PRICE_WINDOW,
        on="week_end",
        closed="left",
        min_periods=REGULAR_PRICE_MIN_PERIODS,
    )["model_unit_price"]
    .quantile(REGULAR_PRICE_QUANTILE)
    .rename("regular_price")
    .reset_index()
)

# Align by identifiers 
main_data["_original_row_order"] = np.arange(len(main_data))

main_data = (
    main_data
    .merge(
        rolling_regular_price,
        on=key_columns,
        how="left",
        validate="one_to_one",
        sort=False,
    )
    .sort_values("_original_row_order")
    .drop(columns="_original_row_order")
    .reset_index(drop=True)
)

del rolling_regular_price, duplicates

In [24]:
main_data["discount_depth"] = (
    1 - main_data["model_unit_price"] / main_data["regular_price"]
).clip(lower=0)

main_data["promo_from_discount"] = main_data["discount_depth"].ge(
    DISCOUNT_THRESHOLD
)
main_data["promo_state"] = (
    main_data["promo_recorded"]
    | main_data["promo_from_discount"]
)

assert main_data["discount_depth"].dropna().ge(0).all()
assert (
    main_data.loc[main_data["promo_recorded"], "promo_state"]
).all()

In [25]:
grouped = main_data.groupby(
    ["store", "upc"],
    observed=True,
    sort=False,
)

main_data["previous_week"] = grouped["week"].shift(1)
main_data["previous_promo"] = (
    grouped["promo_state"].shift(1).astype("boolean")
)

consecutive_previous_week = (
    main_data["week"] - main_data["previous_week"]
).eq(1)

main_data["post_promo"] = (
    consecutive_previous_week
    & main_data["previous_promo"].fillna(False)
    & ~main_data["promo_state"]
)

assert not (
    main_data["post_promo"] & main_data["promo_state"]
).any()

In [26]:
main_data["pricing_state"] = pd.Categorical(
    np.select(
        [main_data["promo_state"], main_data["post_promo"]],
        ["promotion", "post_promotion"],
        default="regular",
    ),
    categories=["regular", "promotion", "post_promotion"],
)

main_data["log_price"] = np.log(
    main_data["model_unit_price"].where(
        main_data["model_unit_price"].gt(0)
    )
)
main_data["scaled_time_trend"] = main_data["time_trend"] / 100

# Integer identifier for store–UPC fixed effects.
main_data["store_upc"] = (
    main_data.groupby(["store", "upc"], observed=True, sort=False)
    .ngroup()
    .astype("int32")
)

## 7. Build the demand estimation sample

The final estimation sample is created after all variables based on earlier weeks have been constructed. UPC 317 is excluded because it refers to a Tony the Tiger T-shirt rather than a cereal product. Product names are stored in a separate lookup table to keep the main dataset smaller.

In [27]:
product_lookup = (
    main_data[["upc", "descrip", "size"]]
    .drop_duplicates(subset="upc")
    .sort_values("upc")
    .reset_index(drop=True)
)

excluded_products = product_lookup.loc[
    product_lookup["upc"].isin(NON_CEREAL_UPCS)
]
display(excluded_products)

assert set(excluded_products["upc"]) == NON_CEREAL_UPCS
assert excluded_products["descrip"].astype("string").str.contains(
    "T-SH", case=False, na=False
).all()

main_data["valid_cereal_product"] = ~main_data["upc"].isin(
    NON_CEREAL_UPCS
)

,upc,descrip,size
0,317,$TONY THE TIGER T-SH,ASST


In [28]:
demand_columns = [
    # Panel identifiers
    "store", "upc", "store_upc", "week", "week_start", "week_end",

    # Outcome and price
    "move", "model_unit_price", "unit_price_observed", "log_price",
    "regular_price", "discount_depth",

    # Price provenance
    "price_source", "price_imputed", "tier_week_price_donors",
    "price_tier",

    # Promotion state
    "sale", "promo_recorded", "promo_from_discount", "promo_state",
    "post_promo", "pricing_state",

    # Calendar controls
    "calendar_month", "scaled_time_trend", "thanksgiving_week",
    "christmas_week", "new_year_week", "easter_week",

    # Observed margin for later cost reconstruction
    "gross_margin_pct_observed",
]

In [29]:
demand_sample_mask = (
    main_data["valid_cereal_product"]
    & main_data["move"].ge(0)
    & main_data["model_unit_price"].gt(0)
    & main_data["regular_price"].gt(0)
    & main_data["discount_depth"].notna()
    & np.isfinite(main_data["log_price"])
)

# Create the eligible estimation sample.
demand_data = main_data.loc[
    demand_sample_mask,
    demand_columns,
].copy(deep=False)

assert demand_data["price_source"].isin(
    ["observed", "tier_median_imputed"]
).all()
assert demand_data["price_imputed"].eq(
    demand_data["unit_price_observed"].isna()
).all()
assert demand_data.loc[
    demand_data["price_imputed"], "gross_margin_pct_observed"
].isna().all()
# Delete main_data.
del main_data

In [30]:
eligible_sample_summary = pd.Series(
    {
        "rows": len(demand_data),
        "stores": demand_data["store"].nunique(),
        "products": demand_data["upc"].nunique(),
        "store_upc_pairs": demand_data["store_upc"].nunique(),
        "weeks": demand_data["week"].nunique(),
        "zero_sales_share": demand_data["move"].eq(0).mean(),
        "imputed_price_share": demand_data["price_imputed"].mean(),
        "recorded_promo_share": demand_data["promo_recorded"].mean(),
        "combined_promo_share": demand_data["promo_state"].mean(),
        "post_promo_share": demand_data["post_promo"].mean(),
    },
    name="value",
)

display(eligible_sample_summary)

rows                    5.125995e+06
stores                  8.600000e+01
products                4.610000e+02
store_upc_pairs         3.451900e+04
weeks                   3.580000e+02
zero_sales_share        1.214673e-01
imputed_price_share     1.214673e-01
recorded_promo_share    6.594778e-02
combined_promo_share    1.392744e-01
post_promo_share        4.623317e-02
Name: value, dtype: float64

## 8. Identify unusual sales observations

This step flags store-level sales that differ sharply from other stores selling the same product in the same week. The aim is to identify likely recording errors without removing demand changes shared across stores. All flagged observations are reported before exclusion.

In [31]:
demand_model_data, sales_outlier_audit, outlier_summary = (
    detect_cross_store_sales_outliers(
        demand_data=demand_data,
        product_lookup=product_lookup,
        demand_columns=demand_columns,
        config=sample_config,
    )
)

In [32]:
display(outlier_summary)

flagged_rows          50.00000
flagged_share          0.00001
flagged_units     188828.00000
flagged_stores        12.00000
Name: value, dtype: float64

In [33]:
display(sales_outlier_audit.head(10))

,store,upc,store_upc,week,week_start,week_end,move,model_unit_price,unit_price_observed,log_price,...,gross_margin_pct_observed,sales_outlier,_row_index,descrip,size,reporting_stores,comparison_median_move,comparison_p95_move,move_to_median,move_to_p95
9,86,3000006560,17013,395,1997-04-03,1997-04-09,17824,1.50,1.50,0.405465,...,-70.20,True,3096520,CAPN CRUNCH JUMBO CR,15 OZ,80,11.0,25.30,1620.363636,704.505929
11,86,3000006610,17015,395,1997-04-03,1997-04-09,8406,1.50,1.50,0.405465,...,-69.77,True,3097142,CAPN CRUNCH CEREAL,16 OZ,80,10.0,20.00,840.600000,420.300000
3,86,1600066110,16876,376,1996-11-21,1996-11-27,10774,0.99,0.99,-0.010050,...,-25.92,True,3072772,WHEATIES,12 OZ,80,15.0,26.80,718.266667,402.014925
1,84,3000006860,16590,150,1992-07-23,1992-07-29,3447,2.67,2.67,0.982078,...,0.71,True,3019920,QKR CRNCHY CORN BRAN,16 OZ,84,7.0,16.90,492.428571,203.964497
7,86,3000006500,17012,395,1997-04-03,1997-04-09,4045,1.50,1.50,0.405465,...,-69.69,True,3096158,QUAKER P.B. CAPTAIN,15 OZ,80,10.0,25.00,404.500000,161.800000
40,100,3800001111,21483,340,1996-03-14,1996-03-20,1614,2.01,2.01,0.698135,...,-59.57,True,3919992,KELLOGGS COCOA KRISP,15 OZ,84,13.0,25.90,124.153846,62.316602
49,133,3800000520,33554,131,1992-03-12,1992-03-18,4159,2.45,2.45,0.896088,...,14.24,True,6143249,KELLOGGS RICE KRISPI,15 OZ,85,35.5,67.25,117.154930,61.843866
48,132,3000006610,33128,365,1996-09-05,1996-09-11,1164,0.99,0.99,-0.010050,...,-54.84,True,6067675,CAPN CRUNCH CEREAL,16 OZ,84,9.0,22.80,129.333333,51.052632
46,124,3800001520,30625,315,1995-09-21,1995-09-27,1616,1.39,1.39,0.329304,...,-46.58,True,5616627,KELLOGG'S FROSTED FL,20 OZ,85,15.0,31.85,107.733333,50.737834
42,103,3000006610,22705,365,1996-09-05,1996-09-11,1122,0.99,0.99,-0.010050,...,-21.61,True,4145830,CAPN CRUNCH CEREAL,16 OZ,84,9.0,22.80,124.666667,49.210526


## 9. Final quality assurance


In [34]:
required_columns = [
    "store", "upc", "store_upc", "week", "move",
    "model_unit_price", "log_price", "regular_price",
    "discount_depth", "promo_recorded", "post_promo",
    "calendar_month", "scaled_time_trend",
    "thanksgiving_week", "christmas_week",
    "new_year_week", "easter_week",
]

missing_required = demand_model_data[required_columns].isna().sum()
display(missing_required.rename("missing_values"))

assert missing_required.eq(0).all()
assert not demand_model_data.duplicated(
    ["store", "upc", "week"]
).any()
assert demand_model_data["move"].ge(0).all()
assert demand_model_data["model_unit_price"].gt(0).all()
assert demand_model_data["regular_price"].gt(0).all()
assert demand_model_data["discount_depth"].ge(0).all()
assert demand_model_data["calendar_month"].between(1, 12).all()

store                0
upc                  0
store_upc            0
week                 0
move                 0
model_unit_price     0
log_price            0
regular_price        0
discount_depth       0
promo_recorded       0
post_promo           0
calendar_month       0
scaled_time_trend    0
thanksgiving_week    0
christmas_week       0
new_year_week        0
easter_week          0
Name: missing_values, dtype: int64

In [35]:
assert np.isfinite(demand_model_data["log_price"]).all()
assert np.allclose(
    demand_model_data["log_price"],
    np.log(demand_model_data["model_unit_price"]),
)
assert not demand_model_data["week"].isin(EXCLUDED_WEEKS).any()
assert not demand_model_data["upc"].isin(NON_CEREAL_UPCS).any()
assert not (
    demand_model_data["post_promo"]
    & demand_model_data["promo_state"]
).any()
assert demand_model_data["price_imputed"].eq(
    demand_model_data["unit_price_observed"].isna()
).all()
assert demand_model_data.loc[
    demand_model_data["price_imputed"], "gross_margin_pct_observed"
].isna().all()

In [36]:
final_sample_summary = pd.Series(
    {
        "rows": len(demand_model_data),
        "stores": demand_model_data["store"].nunique(),
        "products": demand_model_data["upc"].nunique(),
        "store_upc_pairs": demand_model_data["store_upc"].nunique(),
        "weeks": demand_model_data["week"].nunique(),
        "zero_sales_share": demand_model_data["move"].eq(0).mean(),
        "imputed_price_share": demand_model_data["price_imputed"].mean(),
        "recorded_promo_share": demand_model_data["promo_recorded"].mean(),
        "post_promo_share": demand_model_data["post_promo"].mean(),
        "excluded_sales_outliers": outlier_summary["flagged_rows"],
    },
    name="value",
)

display(final_sample_summary)

rows                       5.125945e+06
stores                     8.600000e+01
products                   4.610000e+02
store_upc_pairs            3.451900e+04
weeks                      3.580000e+02
zero_sales_share           1.214685e-01
imputed_price_share        1.214685e-01
recorded_promo_share       6.594199e-02
post_promo_share           4.623362e-02
excluded_sales_outliers    5.000000e+01
Name: value, dtype: float64

In [37]:
# Descriptive summary
descriptive_columns = [
    "move", "model_unit_price", "regular_price", "discount_depth",
    "log_price", "scaled_time_trend", "gross_margin_pct_observed",
]

descriptive_summary = demand_model_data[
    descriptive_columns
].describe(
    percentiles=[0.50, 0.90, 0.95, 0.99, 0.999]
).T

display(descriptive_summary)

,count,mean,std,min,50%,90%,95%,99%,99.9%,max
move,5125945.0,16.863153,38.566187,0.000000,11.000000,31.000000,43.000000,117.000000,522.000000,6126.000000
model_unit_price,5125945.0,3.111305,0.784072,0.050000,3.150000,4.050000,4.350000,4.860000,6.690000,9.710000
regular_price,5125945.0,3.178384,0.776419,0.250000,3.190000,4.090000,4.450000,4.890000,6.980000,7.490000
discount_depth,5125945.0,0.022416,0.071707,0.000000,0.000000,0.066890,0.141672,0.426513,0.563877,0.979123
log_price,5125945.0,1.094567,0.313853,-2.995732,1.147402,1.398717,1.470176,1.581038,1.900614,2.273156
scaled_time_trend,5125945.0,1.974435,1.131449,0.040000,1.890000,3.620000,3.810000,3.950000,3.980000,3.980000
gross_margin_pct_observed,4503304.0,17.284788,9.763018,-99.690000,16.600000,26.640000,33.950000,47.970000,62.580000,99.990000


## Save processed datasets

The processed datasets and audit summaries are saved for use in the downstream notebooks.

### 10.1 Save Parquet datasets


In [38]:
DEMAND_DATA_PATH = PROCESSED_DIR / "cereal_demand_data.parquet"
DEMAND_MODEL_DATA_PATH = PROCESSED_DIR / "cereal_demand_model_data.parquet"
PRODUCT_LOOKUP_PATH = PROCESSED_DIR / "cereal_product_lookup.parquet"
SALES_OUTLIER_PATH = PROCESSED_DIR / "cereal_sales_outliers.parquet"

parquet_outputs = {
    "eligible demand sample": (DEMAND_DATA_PATH, demand_data),
    "cleaned demand sample": (DEMAND_MODEL_DATA_PATH, demand_model_data),
    "product lookup": (PRODUCT_LOOKUP_PATH, product_lookup),
    "sales-anomaly audit": (SALES_OUTLIER_PATH, sales_outlier_audit),
}
parquet_audit = write_parquet_outputs(
    parquet_outputs,
    compression=PARQUET_COMPRESSION,
)

### 10.2 Verify Parquet outputs


In [39]:
display(parquet_audit)

,output,path,rows_expected,rows_written,columns_expected,columns_written
0,eligible demand sample,C:\Users\janza\Desktop\dynamic_promotion_plann...,5125995,5125995,29,29
1,cleaned demand sample,C:\Users\janza\Desktop\dynamic_promotion_plann...,5125945,5125945,29,29
2,product lookup,C:\Users\janza\Desktop\dynamic_promotion_plann...,490,490,3,3
3,sales-anomaly audit,C:\Users\janza\Desktop\dynamic_promotion_plann...,50,50,38,38


### 10.3 Export audit summaries as CSV


In [40]:
final_sample_summary.to_csv(
    TABLE_DIR / "cereal_demand_sample_summary.csv",
    header=True,
)
validation_summary.to_csv(
    TABLE_DIR / "price_imputation_validation_summary.csv",
    header=True,
)
validation_by_donors.to_csv(
    TABLE_DIR / "price_imputation_validation_by_donors.csv"
)
validation_by_tier.to_csv(
    TABLE_DIR / "price_imputation_validation_by_tier.csv"
)
outlier_summary.to_csv(
    TABLE_DIR / "sales_outlier_summary.csv",
    header=True,
)
descriptive_summary.to_csv(
    TABLE_DIR / "cereal_demand_descriptive_summary.csv"
)

print("Saved final demand sample:", DEMAND_MODEL_DATA_PATH)
print("Final shape:", demand_model_data.shape)

Saved final demand sample: C:\Users\janza\Desktop\dynamic_promotion_planning_renamed_workflow_0.3.0\data\processed\cereal_demand_model_data.parquet
Final shape: (5125945, 29)
